# Reproduce the final UAV wildlife training configuration

This Colab-ready notebook documents the final Thermal and RGB YOLOv8n runs. The source imagery and trained weights are not bundled; provide YOLO-format datasets under `/content/datasets`.

In [ ]:
!pip install -q ultralytics

from collections import Counter
from pathlib import Path
import torch
from ultralytics import YOLO

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. Locate and inspect the datasets

Empty `.txt` files are retained as valid negative frames. The check below reports image/label parity and class counts before GPU time is spent.

In [ ]:
DATA_ROOT = Path('/content/datasets')
CLASS_NAMES = {0: 'Human', 1: 'Vehicle', 2: 'Deer', 3: 'Hare', 4: 'Dog', 5: 'Duck'}

for modality in ('YOLO_Thermal', 'YOLO_RGB'):
    root = DATA_ROOT / modality
    assert (root / 'data.yaml').is_file(), f'Missing {root / "data.yaml"}'
    print(f'\n{modality}')
    for split in ('train', 'val'):
        images = list((root / 'images' / split).glob('*'))
        labels = list((root / 'labels' / split).glob('*.txt'))
        counts = Counter()
        negatives = 0
        for label in labels:
            lines = [line for line in label.read_text().splitlines() if line.strip()]
            negatives += not lines
            counts.update(int(line.split()[0]) for line in lines)
        print(split, {'images': len(images), 'labels': len(labels), 'negative_frames': negatives})
        print({CLASS_NAMES[key]: value for key, value in sorted(counts.items())})

## 2. Train the reported final configurations

The two streams are optimized independently because their native resolution and image statistics differ.

In [ ]:
FINAL_RUNS = {
    'thermal': {'data': DATA_ROOT / 'YOLO_Thermal/data.yaml', 'imgsz': 640},
    'rgb': {'data': DATA_ROOT / 'YOLO_RGB/data.yaml', 'imgsz': 1280},
}

for modality, config in FINAL_RUNS.items():
    model = YOLO('yolov8n.pt')
    model.train(
        data=str(config['data']),
        epochs=100,
        imgsz=config['imgsz'],
        batch=16,
        optimizer='SGD',
        patience=20,
        name=f'{modality}_yolov8n_{config["imgsz"]}px_sgd',
        save=True,
        plots=True,
    )

## 3. Expected evidence and interpretation

Final saved runs reported Thermal mAP@50 = **84.70%** at 640 px and RGB mAP@50 = **85.93%** at 1280 px. For the experiment sweep, use `imgsz` values 640, 960, and 1280 and compare `SGD` with `AdamW`.

The OOD count result is a sequence-level proxy, not object-level recall:

In [ ]:
detections, annotations = 555, 572
coverage = detections / annotations
print(f'OOD count coverage: {coverage:.2%}')
print('Do not interpret this as object-level recall without one-to-one matching.')